# SpecTNT — End-to-End Colab Training

Trains all 3 model configs (HarmonicCNN, SpecTNT, SpecTNT+CTL) across 4 folds.

**Pipeline:** Mount Drive → Clone repos → Download audio (persisted on Drive) → Precompute features → Train → Show results

Audio, features, and training outputs are stored on Google Drive so they persist across sessions.

In [1]:
import os, sys, json, time, csv, glob, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd

print("Environment ready")

Environment ready


## 0. Mount Google Drive

Heavy data (audio, features, training outputs) lives here to persist across sessions.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = "/content/drive/MyDrive/spectnt_data/harmonixset"
DRIVE_OUT  = "/content/drive/MyDrive/spectnt_data/outputs"

os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f"Drive data dir: {DRIVE_DATA}")
print(f"Drive out dir : {DRIVE_OUT}")

Mounted at /content/drive
Drive data dir: /content/drive/MyDrive/spectnt_data/harmonixset
Drive out dir : /content/drive/MyDrive/spectnt_data/outputs


## 1. Clone Repositories & Install Dependencies

In [5]:
REPO = "https://github.com/fharookshaik/BTU_RM_So26.git"
if not os.path.isdir("BTU_RM_So26"):
    !git clone {REPO}
%cd BTU_RM_So26

!pip install -q . 2>&1 | tail -5

# Clone harmonixset annotations (tiny, always fresh)
os.makedirs("data", exist_ok=True)
if not os.path.isdir("data/harmonixset"):
    !git clone https://github.com/urinieto/harmonixset.git data/harmonixset

# Symlink audio/, features/, outputs/ to Drive so they persist
for sub in ["audio", "features"]:
    src = os.path.join(DRIVE_DATA, sub)
    dst = os.path.join("data/harmonixset", sub)
    os.makedirs(src, exist_ok=True)
    if not os.path.islink(dst):
        !ln -s "{src}" "{dst}"

if not os.path.islink("outputs"):
    !ln -s "{DRIVE_OUT}" outputs

# Verify annotations
seg_dir = "data/harmonixset/dataset/segments"
n_seg = len(os.listdir(seg_dir)) if os.path.isdir(seg_dir) else 0
print(f"Annotations: {n_seg} segment files")

Cloning into 'BTU_RM_So26'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 110 (delta 43), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 11.45 MiB | 19.41 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/BTU_RM_So26
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 82.7 MB/s eta 0:00:00
Cloning into 'data/harmonixset'...
remote: Enumerating objects: 11206, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 11206 (delta 103), reused 154 (delta 100), pack-reused 11028 (from 1)
Receiving objects: 100% (11206/11206), 12.73 MiB | 12.12 MiB

## 2. Download Audio (Parallel + Resumable)

Downloads `.wav` files from YouTube URLs to **Google Drive** (persists across sessions).
Skips existing files so re-running resumes where it left off.

In [4]:
AUDIO_DIR = "data/harmonixset/audio"
URL_CSV   = "data/harmonixset/dataset/youtube_urls.csv"
FAILED_LOG = "download_failed.json"

os.makedirs(AUDIO_DIR, exist_ok=True)

# Count existing files
existing = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(AUDIO_DIR, "*.wav"))}
print(f"Already on Drive: {len(existing)} .wav files")

# Load previous failures
failed_already = set()
if os.path.exists(FAILED_LOG):
    failed_already = set(json.load(open(FAILED_LOG)))
    print(f"Previously failed: {len(failed_already)}")

# Build queue
entries = []
with open(URL_CSV) as f:
    for row in csv.DictReader(f):
        fid = row["File"]
        if fid not in existing and fid not in failed_already:
            entries.append(row)

total_to_download = len(entries)
print(f"Need to download: {total_to_download}")

if total_to_download == 0:
    print("All files already on Drive!")
else:
    def download_one(row):
        fid, url = row["File"], row["URL"]
        out_path = os.path.join(AUDIO_DIR, f"{fid}.%(ext)s")
        try:
            subprocess.run([
                "yt-dlp", "-x", "--audio-format", "wav",
                "--audio-quality", "0",
                "--postprocessor-args", "ffmpeg:-ac 1 -ar 16000",
                "-o", out_path,
                "--no-playlist", "--quiet", "--no-warnings",
                url,
            ], check=True, capture_output=True, timeout=120)
            return (fid, True, None)
        except Exception as e:
            return (fid, False, str(e))

    failed = []
    total_all = len(existing) + len(failed_already) + len(entries)
    done_count = len(existing) + len(failed_already)
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures = {pool.submit(download_one, e): e for e in entries}
        for fut in as_completed(futures):
            fid, ok, err = fut.result()
            done_count += 1
            if ok:
                print(f"[{done_count}/{total_all}] {fid} OK")
            else:
                print(f"[{done_count}/{total_all}] {fid} FAILED: {err[:60]}")
                failed.append(fid)

    json.dump(failed, open(FAILED_LOG, "w"))
    print(f"\nDone. Failed: {len(failed)}. Re-run this cell to retry.")

Already on Drive: 0 .wav files


FileNotFoundError: [Errno 2] No such file or directory: 'data/harmonixset/dataset/youtube_urls.csv'

In [ ]:
audio_count = len(glob.glob(os.path.join(AUDIO_DIR, "*.wav")))
yt_count = sum(1 for _ in open(URL_CSV)) - 1
print(f"Audio files on Drive: {audio_count} / {yt_count} total tracks")
if audio_count < yt_count * 0.5:
    print("WARNING: Less than 50% audio. Training on what's available.")

## 3. Precompute Features

Computes mel-spectrogram features (`.npy`) from audio. Stores on Drive — persists across sessions.

In [ ]:
from src.features import precompute_features
from src.data_utils import get_songs_with_audio

songs = get_songs_with_audio()
print(f"Songs with audio: {len(songs)}")

paths = precompute_features(songs, force=False)
feature_count = sum(1 for p in paths.values() if os.path.exists(p))
print(f"Features ready: {feature_count} / {len(songs)}")

if feature_count < len(songs):
    paths = precompute_features(songs, force=True)
    feature_count = sum(1 for p in paths.values() if os.path.exists(p))
    print(f"Features after force: {feature_count} / {len(songs)}")

## 4. Train All Experiments

Runs 3 models × 4 folds = 12 training sessions. Checkpoints & results save to Drive.

In [ ]:
models = ["harmonic_cnn", "spectnt", "spectnt_ctl"]
folds  = list(range(4))

for model in models:
    for fold in folds:
        print(f"\n{'='*60}\nStarting: {model}, Fold {fold}\n{'='*60}")
        !python main.py --model {model} --fold {fold}


## 5. Results Summary

Displays per-model, per-fold metrics from Drive-persisted results.

In [ ]:
results_path = "outputs/results/results.json"
if not os.path.exists(results_path):
    print("No results found yet. Complete Step 4 first.")
else:
    all_results = json.load(open(results_path))
    for model_name, data in all_results.items():
        print(f"\n{'='*60}")
        print(f"{model_name}")
        print(f"{'='*60}")
        df = pd.DataFrame(data["per_fold"])
        df.index = [f"Fold {i}" for i in range(len(df))]
        print(df.to_string())
        print(f"\nAverage: {data['average']}")